In [1]:
# # !pip install scikit-learn nltk pandas matplotlib

import pandas as pd
import numpy as np
import nltk
import re
import string

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt')

[nltk_data] Error loading stopwords: <urlopen error [Errno 11001]
[nltk_data]     getaddrinfo failed>
[nltk_data] Error loading wordnet: <urlopen error [Errno 11001]
[nltk_data]     getaddrinfo failed>
[nltk_data] Error loading punkt: <urlopen error [Errno 11001]
[nltk_data]     getaddrinfo failed>


False

In [2]:
corpus = [
    "The cat sat on the mat.",
    "The dog played with the cat.",
    "Dogs and cats are great pets.",
    "I love my pet dog and my pet cat."
]

df = pd.DataFrame({'text': corpus})
df

,text
0,The cat sat on the mat.
1,The dog played with the cat.
2,Dogs and cats are great pets.
3,I love my pet dog and my pet cat.


In [3]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)          # remove punctuation/numbers
    tokens = text.split()
    tokens = [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words]
    return ' '.join(tokens)

df['clean_text'] = df['text'].apply(preprocess_text)
df

LookupError: 
**********************************************************************
  Resource [93mstopwords[0m not found.
  Please use the NLTK Downloader to obtain the resource:

  [31m>>> import nltk
  >>> nltk.download('stopwords')
  [0m
  For more information see: https://www.nltk.org/data.html

  Attempted to load [93mcorpora/stopwords[0m

  Searched in:
    - 'C:\\Users\\mahab/nltk_data'
    - 'C:\\Users\\mahab\\anaconda3\\nltk_data'
    - 'C:\\Users\\mahab\\anaconda3\\share\\nltk_data'
    - 'C:\\Users\\mahab\\anaconda3\\lib\\nltk_data'
    - 'C:\\Users\\mahab\\AppData\\Roaming\\nltk_data'
    - 'C:\\nltk_data'
    - 'D:\\nltk_data'
    - 'E:\\nltk_data'
**********************************************************************


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

cv = CountVectorizer()
bow_matrix = cv.fit_transform(df['clean_text'])

bow_df = pd.DataFrame(
    bow_matrix.toarray(),
    columns=cv.get_feature_names_out(),
    index=[f"Doc{i+1}" for i in range(len(df))]
)

print("Vocabulary:", cv.vocabulary_)
bow_df

In [ ]:
# Each row = a document, each column = a word, each cell = word frequency
print("Shape of BoW matrix:", bow_matrix.shape)
print("\nTotal occurrences of each word across corpus:")
print(bow_df.sum().sort_values(ascending=False))

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer()
tfidf_matrix = tfidf.fit_transform(df['clean_text'])

tfidf_df = pd.DataFrame(
    tfidf_matrix.toarray(),
    columns=tfidf.get_feature_names_out(),
    index=[f"Doc{i+1}" for i in range(len(df))]
)

tfidf_df.round(3)

In [ ]:
import math
from collections import Counter

def compute_tf(doc_tokens):
    tf_dict = {}
    total_terms = len(doc_tokens)
    counts = Counter(doc_tokens)
    for word, count in counts.items():
        tf_dict[word] = count / total_terms
    return tf_dict

def compute_idf(all_docs_tokens):
    N = len(all_docs_tokens)
    idf_dict = {}
    all_words = set(word for doc in all_docs_tokens for word in doc)
    for word in all_words:
        containing_docs = sum(1 for doc in all_docs_tokens if word in doc)
        idf_dict[word] = math.log(N / containing_docs) + 1  # smoothed
    return idf_dict

tokenized_docs = [doc.split() for doc in df['clean_text']]
idf_scores = compute_idf(tokenized_docs)

print("Manual IDF scores:")
for word, score in sorted(idf_scores.items(), key=lambda x: -x[1]):
    print(f"{word}: {score:.4f}")

In [ ]:
def get_top_words(doc_index, top_n=3):
    row = tfidf_df.iloc[doc_index]
    return row.sort_values(ascending=False).head(top_n)

for i in range(len(df)):
    print(f"\nTop words in Doc{i+1}:")
    print(get_top_words(i))

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
tfidf_df.T.plot(kind='bar', figsize=(12, 6))
plt.title("TF-IDF Scores per Document")
plt.ylabel("TF-IDF Score")
plt.xlabel("Words")
plt.xticks(rotation=45)
plt.legend(title="Documents")
plt.tight_layout()
plt.show()